# MonkeyNeuralForecasting — Data Exploration

This notebook explores the raw data for both monkeys (Affi and Beignet).

Key goals:
1. Understand data shape, scale, and quality
2. Visualize raw LMP signals and power band features
3. Compute inter-channel correlation structure
4. Check for rotational dynamics in PCA space
5. Evaluate persistence baseline MSE
6. Identify interesting observations for model design

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 100

from src.data.dataset import load_npz, compute_normalization_stats, normalize, split_context_target, DATASET_DIR, MONKEY_FILES
from src.models.baselines.persistence import PersistenceModel
from src.evaluation.metrics import mse, per_timestep_mse, per_channel_mse
from src.visualization.plots import (
    plot_correlation_matrix, plot_pca_trajectories, plot_feature_statistics
)

print('Setup complete.')

## 1. Load Raw Data

In [ ]:
# Load raw data for both monkeys
affi_raw = load_npz(os.path.join(DATASET_DIR, MONKEY_FILES['affi']['train']))
beignet_raw = load_npz(os.path.join(DATASET_DIR, MONKEY_FILES['beignet']['train']))

print(f'Affi shape:   {affi_raw.shape}  dtype={affi_raw.dtype}')
print(f'Beignet shape: {beignet_raw.shape}  dtype={beignet_raw.dtype}')
print()
print(f'Affi   — min={affi_raw.min():.1f}, max={affi_raw.max():.1f}, mean={affi_raw.mean():.1f}, std={affi_raw.std():.1f}')
print(f'Beignet — min={beignet_raw.min():.1f}, max={beignet_raw.max():.1f}, mean={beignet_raw.mean():.1f}, std={beignet_raw.std():.1f}')

## 2. Signal Visualization — Raw LMP Traces

In [ ]:
def plot_raw_signals(data, monkey, n_channels=6, n_trials=3):
    """Plot raw LMP traces for a few channels and trials."""
    t = np.arange(data.shape[1]) * 30  # ms
    channels = np.linspace(0, data.shape[2]-1, n_channels, dtype=int)
    
    fig, axes = plt.subplots(n_trials, n_channels, figsize=(3*n_channels, 2.5*n_trials))
    
    for trial in range(n_trials):
        for i, ch in enumerate(channels):
            ax = axes[trial, i]
            lmp = data[trial, :, ch, 0]
            ax.plot(t, lmp, 'steelblue', linewidth=1.5)
            ax.axvline(300, color='red', linestyle='--', alpha=0.5, label='Context/Pred boundary')
            if trial == 0:
                ax.set_title(f'Ch {ch}', fontsize=9)
            if i == 0:
                ax.set_ylabel(f'Trial {trial}\nLMP (μV)', fontsize=8)
            ax.set_xlabel('Time (ms)', fontsize=8)
            ax.grid(True, alpha=0.3)
    
    fig.suptitle(f'{monkey} — Raw LMP Signals (red line = context/prediction boundary)', fontsize=11)
    plt.tight_layout()
    plt.show()

plot_raw_signals(affi_raw, 'Affi')
plot_raw_signals(beignet_raw, 'Beignet')

## 3. Feature Distributions

In [ ]:
feature_names = ['LMP', '0.5-4Hz', '4-8Hz', '8-12Hz', '12-25Hz', '25-50Hz', '50-100Hz', '100-200Hz', '200-400Hz']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, monkey in zip(axes, [affi_raw, beignet_raw], ['Affi', 'Beignet']):
    flat = data[:, :10, :, :].reshape(-1, 9)  # context steps only
    ax.boxplot([flat[:, f] for f in range(9)], labels=feature_names, showfliers=False)
    ax.set_title(f'{monkey} — Feature Distributions (context steps)')
    ax.set_ylabel('Value (μV)')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Normalize Data

In [ ]:
# Compute normalization stats from training data
affi_stats = compute_normalization_stats(affi_raw)
beignet_stats = compute_normalization_stats(beignet_raw)

affi_norm = normalize(affi_raw, affi_stats)
beignet_norm = normalize(beignet_raw, beignet_stats)

print(f'Affi normalized — min={affi_norm.min():.3f}, max={affi_norm.max():.3f}, mean={affi_norm.mean():.3f}')
print(f'Beignet normalized — min={beignet_norm.min():.3f}, max={beignet_norm.max():.3f}, mean={beignet_norm.mean():.3f}')

## 5. Persistence Baseline MSE

In [ ]:
persistence = PersistenceModel()

for data_norm, monkey in [(affi_norm, 'Affi'), (beignet_norm, 'Beignet')]:
    X, Y = split_context_target(data_norm)
    pred = persistence.predict_batch(X)
    
    total_mse = mse(pred, Y)
    ts_mse = per_timestep_mse(pred, Y)
    ch_mse = per_channel_mse(pred, Y)
    
    print(f'\n{monkey} Persistence Baseline:')
    print(f'  MSE: {total_mse:.6f}')
    print(f'  Per-timestep MSE: t=0: {ts_mse[0]:.4f}, t=5: {ts_mse[5]:.4f}, t=9: {ts_mse[9]:.4f}')
    print(f'  Per-channel MSE: min={ch_mse.min():.4f}, max={ch_mse.max():.4f}, mean={ch_mse.mean():.4f}')

## 6. Inter-Channel Correlation Matrix

In [ ]:
def compute_correlation(data, max_ch=50):
    """Compute inter-channel correlation matrix from LMP."""
    lmp = data[:, :10, :max_ch, 0].reshape(-1, min(data.shape[2], max_ch))
    return np.corrcoef(lmp.T)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, data, monkey in zip(axes, [affi_norm, beignet_norm], ['Affi', 'Beignet']):
    max_ch = min(50, data.shape[2])
    corr = compute_correlation(data, max_ch)
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
    plt.colorbar(im, ax=ax, label='Pearson r')
    ax.set_title(f'{monkey} — Inter-Channel Correlation (first {max_ch} channels)')
    ax.set_xlabel('Channel')
    ax.set_ylabel('Channel')

plt.tight_layout()
plt.show()

## 7. PCA of Population Trajectories

In [ ]:
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, data, monkey in zip(axes, [affi_norm, beignet_norm], ['Affi', 'Beignet']):
    lmp = data[:50, :10, :, 0]  # (50, 10, C)
    N, T, C = lmp.shape
    flat = lmp.reshape(N * T, C)
    pca = PCA(n_components=2)
    proj = pca.fit_transform(flat).reshape(N, T, 2)
    
    cmap = plt.cm.viridis
    for i in range(N):
        color = cmap(i / N)
        ax.plot(proj[i, :, 0], proj[i, :, 1], alpha=0.4, color=color, linewidth=1)
        ax.plot(proj[i, 0, 0], proj[i, 0, 1], 'o', color=color, markersize=4)
        ax.plot(proj[i, -1, 0], proj[i, -1, 1], 's', color=color, markersize=4)
    
    var = pca.explained_variance_ratio_
    ax.set_xlabel(f'PC1 ({var[0]:.1%} var)')
    ax.set_ylabel(f'PC2 ({var[1]:.1%} var)')
    ax.set_title(f'{monkey} — Population Trajectories (PCA, circle=start, square=end)')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Check total variance explained by first 20 PCs
for data, monkey in [(affi_norm, 'Affi'), (beignet_norm, 'Beignet')]:
    lmp = data[:, :10, :, 0].reshape(-1, data.shape[2])
    pca = PCA(n_components=min(20, data.shape[2]))
    pca.fit(lmp)
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    for n_pcs in [2, 5, 10, 20]:
        print(f'{monkey}: {n_pcs} PCs explain {cumvar[n_pcs-1]:.1%} of variance')
    print()

## 8. Temporal Autocorrelation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, data, monkey in zip(axes, [affi_norm, beignet_norm], ['Affi', 'Beignet']):
    # Compute mean autocorrelation across channels
    lmp = data[:, :, :, 0]  # (N, 20, C)
    autocorr_by_lag = []
    for lag in range(1, 10):
        # Correlation between t and t+lag, across all N and C
        t_a = lmp[:, :-lag, :].reshape(-1)
        t_b = lmp[:, lag:, :].reshape(-1)
        r = np.corrcoef(t_a, t_b)[0, 1]
        autocorr_by_lag.append(r)
    
    lags = np.arange(1, 10) * 30  # ms
    ax.plot(lags, autocorr_by_lag, 'o-', linewidth=2, markersize=6)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Lag (ms)')
    ax.set_ylabel('Pearson Correlation')
    ax.set_title(f'{monkey} — LMP Temporal Autocorrelation')
    ax.set_ylim(-0.1, 1.0)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Observations Summary

Fill in findings here and copy to `knowledge/data_exploration/eda_report.md`:

- **Rotational dynamics**: Do PCA trajectories show circular structure?
- **Dimensionality**: How many PCs are needed to explain 90% variance?
- **Autocorrelation**: How quickly does correlation decay with lag?
- **Persistence MSE**: What is the baseline MSE? (Any learned model must beat this)
- **Correlation structure**: Are channels locally correlated?
- **Feature distributions**: Are power bands Gaussian? Are there outliers?